In [37]:

import sqlite3

connection = sqlite3.connect("retail.db")
cursor = connection.cursor()

print("Retail database created successfully!")


Retail database created successfully!


In [38]:

cursor.execute("""
CREATE TABLE IF NOT EXISTS customers (
    customer_id TEXT PRIMARY KEY,
    customer_name TEXT NOT NULL,
    city TEXT,
    signup_date TEXT
);
""")

connection.commit()

print("Customers table created successfully!")


Customers table created successfully!


In [39]:

customers_data = [
    ("C001", "Ravi", "Hyderabad", "2026-01-10"),
    ("C002", "Priya", "Bengaluru", "2026-02-15"),
    ("C003", "Arun", "Chennai", "2026-03-20"),
    ("C004", "Sneha", "Hyderabad", "2026-04-05"),
    ("C005", "Kiran", "Mumbai", "2026-05-12")
]

cursor.executemany("""
INSERT OR IGNORE INTO customers
(customer_id, customer_name, city, signup_date)
VALUES (?, ?, ?, ?);
""", customers_data)

connection.commit()

print("Customer data inserted successfully!")


Customer data inserted successfully!


In [40]:

cursor.execute("SELECT * FROM customers;")

rows = cursor.fetchall()

for row in rows:
    print(row)


('C001', 'Ravi', 'Hyderabad', '2026-01-10')
('C002', 'Priya', 'Bengaluru', '2026-02-15')
('C003', 'Arun', 'Chennai', '2026-03-20')
('C004', 'Sneha', 'Hyderabad', '2026-04-05')
('C005', 'Kiran', 'Mumbai', '2026-05-12')


In [41]:

cursor.execute("""
CREATE TABLE IF NOT EXISTS products (
    product_id TEXT PRIMARY KEY,
    product_name TEXT NOT NULL,
    category TEXT,
    price REAL NOT NULL,
    stock_quantity INTEGER
);
""")

connection.commit()

print("Products table created successfully!")


Products table created successfully!


In [42]:

products_data = [
    ("P001", "Mobile", "Electronics", 15000, 50),
    ("P002", "Laptop", "Electronics", 45000, 20),
    ("P003", "Headphones", "Accessories", 2000, 100),
    ("P004", "Tablet", "Electronics", 12000, 30),
    ("P005", "Keyboard", "Accessories", 1500, 75),
    ("P006", "Mouse", "Accessories", 800, 120)
]

cursor.executemany("""
INSERT OR IGNORE INTO products
(product_id, product_name, category, price, stock_quantity)
VALUES (?, ?, ?, ?, ?);
""", products_data)

connection.commit()

print("Product records inserted successfully!")


Product records inserted successfully!


In [43]:

cursor.execute("SELECT * FROM products;")

products = cursor.fetchall()

for product in products:
    print(product)


('P001', 'Mobile', 'Electronics', 15000.0, 50)
('P002', 'Laptop', 'Electronics', 45000.0, 20)
('P003', 'Headphones', 'Accessories', 2000.0, 100)
('P004', 'Tablet', 'Electronics', 12000.0, 30)
('P005', 'Keyboard', 'Accessories', 1500.0, 75)
('P006', 'Mouse', 'Accessories', 800.0, 120)


In [44]:

connection.commit()
connection.execute("PRAGMA foreign_keys = ON;")

cursor.execute("""
CREATE TABLE IF NOT EXISTS orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT NOT NULL,
    product_id TEXT NOT NULL,
    quantity INTEGER NOT NULL CHECK (quantity > 0),
    order_date TEXT NOT NULL,
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""")

connection.commit()

print("Orders table created successfully!")


Orders table created successfully!


In [45]:

orders_data = [
    ("O001", "C001", "P001", 2, "2026-06-01"),
    ("O002", "C002", "P002", 1, "2026-06-02"),
    ("O003", "C003", "P003", 3, "2026-06-03"),
    ("O004", "C004", "P004", 2, "2026-06-04"),
    ("O005", "C005", "P005", 4, "2026-06-05")
]

cursor.executemany("""
INSERT OR IGNORE INTO orders
(order_id, customer_id, product_id, quantity, order_date)
VALUES (?, ?, ?, ?, ?);
""", orders_data)

connection.commit()

print("Orders inserted successfully!")


Orders inserted successfully!


In [46]:

cursor.execute("SELECT * FROM orders;")

orders = cursor.fetchall()

for order in orders:
    print(order)


('O001', 'C001', 'P001', 2, '2026-06-01')
('O002', 'C002', 'P002', 1, '2026-06-02')
('O003', 'C003', 'P003', 3, '2026-06-03')
('O004', 'C004', 'P004', 2, '2026-06-04')
('O005', 'C005', 'P005', 4, '2026-06-05')


In [47]:

cursor.execute("""
SELECT
    o.order_id,
    c.customer_name,
    p.product_name,
    o.quantity,
    p.price,
    o.quantity * p.price AS total_amount
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN products p
    ON o.product_id = p.product_id;
""")

for row in cursor.fetchall():
    print(row)


('O001', 'Ravi', 'Mobile', 2, 15000.0, 30000.0)
('O002', 'Priya', 'Laptop', 1, 45000.0, 45000.0)
('O003', 'Arun', 'Headphones', 3, 2000.0, 6000.0)
('O004', 'Sneha', 'Tablet', 2, 12000.0, 24000.0)
('O005', 'Kiran', 'Keyboard', 4, 1500.0, 6000.0)


In [48]:

import pandas as pd

query = """
SELECT
    o.order_id,
    c.customer_name,
    p.product_name,
    o.quantity,
    p.price,
    o.order_date
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN products p
    ON o.product_id = p.product_id;
"""

df = pd.read_sql_query(query, connection)

print("Extracted records:", len(df))
display(df)


Extracted records: 5


,order_id,customer_name,product_name,quantity,price,order_date
0,O001,Ravi,Mobile,2,15000.0,2026-06-01
1,O002,Priya,Laptop,1,45000.0,2026-06-02
2,O003,Arun,Headphones,3,2000.0,2026-06-03
3,O004,Sneha,Tablet,2,12000.0,2026-06-04
4,O005,Kiran,Keyboard,4,1500.0,2026-06-05


In [49]:

# Step 1: Remove duplicate orders
df_clean = df.drop_duplicates(subset=["order_id"]).copy()

# Step 2: Keep valid orders
df_clean = df_clean[
    (df_clean["quantity"] > 0) &
    (df_clean["price"] >= 0)
].copy()

# Step 3: Calculate sales amount
df_clean["total_amount"] = (
    df_clean["quantity"] * df_clean["price"]
)

# Step 4: Check the result
print("Clean records:", len(df_clean))
display(df_clean)


Clean records: 5


,order_id,customer_name,product_name,quantity,price,order_date,total_amount
0,O001,Ravi,Mobile,2,15000.0,2026-06-01,30000.0
1,O002,Priya,Laptop,1,45000.0,2026-06-02,45000.0
2,O003,Arun,Headphones,3,2000.0,2026-06-03,6000.0
3,O004,Sneha,Tablet,2,12000.0,2026-06-04,24000.0
4,O005,Kiran,Keyboard,4,1500.0,2026-06-05,6000.0


In [50]:

df_clean.to_sql(
    "sales_summary",
    connection,
    if_exists="replace",
    index=False
)

print("ETL pipeline completed successfully!")
print("Loaded records:", len(df_clean))


ETL pipeline completed successfully!
Loaded records: 5


In [51]:

result = pd.read_sql_query(
    "SELECT * FROM sales_summary;",
    connection
)

display(result)

print("Total sales:", result["total_amount"].sum())


,order_id,customer_name,product_name,quantity,price,order_date,total_amount
0,O001,Ravi,Mobile,2,15000.0,2026-06-01,30000.0
1,O002,Priya,Laptop,1,45000.0,2026-06-02,45000.0
2,O003,Arun,Headphones,3,2000.0,2026-06-03,6000.0
3,O004,Sneha,Tablet,2,12000.0,2026-06-04,24000.0
4,O005,Kiran,Keyboard,4,1500.0,2026-06-05,6000.0


Total sales: 111000.0


In [52]:

import pandas as pd

raw_orders = pd.DataFrame([
    ["O101", "C001", "P001", 2, 15000],
    ["O102", "C002", "P002", 1, 45000],
    ["O103", "C003", "P003", -3, 2000],
    ["O104", None, "P004", 2, 12000],
    ["O105", "C005", "P005", 4, None],
    ["O101", "C001", "P001", 2, 15000],
    ["O106", "C004", "P006", 3, 800]
], columns=[
    "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "price"
])

print("Total raw records:", len(raw_orders))
display(raw_orders)


Total raw records: 7


,order_id,customer_id,product_id,quantity,price
0,O101,C001,P001,2,15000.0
1,O102,C002,P002,1,45000.0
2,O103,C003,P003,-3,2000.0
3,O104,None,P004,2,12000.0
4,O105,C005,P005,4,NaN
5,O101,C001,P001,2,15000.0
6,O106,C004,P006,3,800.0


In [53]:

# Find duplicate order IDs
duplicate = raw_orders.duplicated(
    subset=["order_id"],
    keep="first"
)

# Find missing required information
missing = raw_orders[
    ["order_id", "customer_id", "product_id", "quantity", "price"]
].isnull().any(axis=1)

# Find invalid quantity or price
invalid_numbers = (
    (raw_orders["quantity"] <= 0) |
    (raw_orders["price"] < 0)
)

# Combine all validation failures
invalid = duplicate | missing | invalid_numbers

# Separate accepted and rejected records
valid_orders = raw_orders[~invalid].copy()
rejected_orders = raw_orders[invalid].copy()

print("Valid records:", len(valid_orders))
print("Rejected records:", len(rejected_orders))


Valid records: 3
Rejected records: 4


In [54]:

valid_orders["total_amount"] = (
    valid_orders["quantity"] * valid_orders["price"]
)

print("Valid orders:")
display(valid_orders)

print("Rejected orders:")
display(rejected_orders)

print(
    "Total valid sales:",
    valid_orders["total_amount"].sum()
)


Valid orders:


,order_id,customer_id,product_id,quantity,price,total_amount
0,O101,C001,P001,2,15000.0,30000.0
1,O102,C002,P002,1,45000.0,45000.0
6,O106,C004,P006,3,800.0,2400.0


Rejected orders:


,order_id,customer_id,product_id,quantity,price
2,O103,C003,P003,-3,2000.0
3,O104,None,P004,2,12000.0
4,O105,C005,P005,4,NaN
5,O101,C001,P001,2,15000.0


Total valid sales: 77400.0


In [55]:

def validate_order(row):
    reasons = []

    if pd.isna(row["order_id"]):
        reasons.append("Missing order ID")

    if pd.isna(row["customer_id"]):
        reasons.append("Missing customer ID")

    if pd.isna(row["product_id"]):
        reasons.append("Missing product ID")

    if pd.isna(row["quantity"]):
        reasons.append("Missing quantity")
    elif row["quantity"] <= 0:
        reasons.append("Invalid quantity")

    if pd.isna(row["price"]):
        reasons.append("Missing price")
    elif row["price"] < 0:
        reasons.append("Negative price")

    return ", ".join(reasons)

raw_orders["rejection_reason"] = raw_orders.apply(
    validate_order, axis=1
)

display(raw_orders)


,order_id,customer_id,product_id,quantity,price,rejection_reason
0,O101,C001,P001,2,15000.0,
1,O102,C002,P002,1,45000.0,
2,O103,C003,P003,-3,2000.0,Invalid quantity
3,O104,None,P004,2,12000.0,Missing customer ID
4,O105,C005,P005,4,NaN,Missing price
5,O101,C001,P001,2,15000.0,
6,O106,C004,P006,3,800.0,


In [56]:

# Identify duplicate order IDs
duplicate_mask = raw_orders.duplicated(
    subset=["order_id"],
    keep="first"
)

# Add rejection reason for duplicates
raw_orders.loc[
    duplicate_mask, "rejection_reason"
] = raw_orders.loc[
    duplicate_mask, "rejection_reason"
].apply(
    lambda reason: (
        reason + ", Duplicate order ID"
        if reason
        else "Duplicate order ID"
    )
)

# Display the results
display(raw_orders[["order_id", "rejection_reason"]])


,order_id,rejection_reason
0,O101,
1,O102,
2,O103,Invalid quantity
3,O104,Missing customer ID
4,O105,Missing price
5,O101,Duplicate order ID
6,O106,


In [57]:

valid_orders = raw_orders[
    raw_orders["rejection_reason"] == ""
].copy()

rejected_orders = raw_orders[
    raw_orders["rejection_reason"] != ""
].copy()

valid_orders["total_amount"] = (
    valid_orders["quantity"] * valid_orders["price"]
)

print("Valid records:", len(valid_orders))
print("Rejected records:", len(rejected_orders))
print("Total valid sales:", valid_orders["total_amount"].sum())

display(rejected_orders)


Valid records: 3
Rejected records: 4
Total valid sales: 77400.0


,order_id,customer_id,product_id,quantity,price,rejection_reason
2,O103,C003,P003,-3,2000.0,Invalid quantity
3,O104,None,P004,2,12000.0,Missing customer ID
4,O105,C005,P005,4,NaN,Missing price
5,O101,C001,P001,2,15000.0,Duplicate order ID


In [58]:

import sqlite3

# Reconnect to the database if necessary
if "connection" not in globals():
    connection = sqlite3.connect("retail.db")

# Save valid records
valid_orders.to_sql(
    "validated_orders",
    connection,
    if_exists="replace",
    index=False
)

# Save rejected records
rejected_orders.to_sql(
    "rejected_orders",
    connection,
    if_exists="replace",
    index=False
)

print("Validated orders saved:", len(valid_orders))
print("Rejected orders saved:", len(rejected_orders))


Validated orders saved: 3
Rejected orders saved: 4


In [59]:

# Create a copy of the original incoming data
bronze_orders = raw_orders.drop(
    columns=["rejection_reason"],
    errors="ignore"
).copy()

# Save the original data into the Bronze table
bronze_orders.to_sql(
    "bronze_orders",
    connection,
    if_exists="replace",
    index=False
)

print("Bronze records:", len(bronze_orders))

display(bronze_orders)


Bronze records: 7


,order_id,customer_id,product_id,quantity,price
0,O101,C001,P001,2,15000.0
1,O102,C002,P002,1,45000.0
2,O103,C003,P003,-3,2000.0
3,O104,None,P004,2,12000.0
4,O105,C005,P005,4,NaN
5,O101,C001,P001,2,15000.0
6,O106,C004,P006,3,800.0


In [60]:

# Create a copy of validated orders
silver_orders = valid_orders.copy()

# Save clean records into the Silver table
silver_orders.to_sql(
    "silver_orders",
    connection,
    if_exists="replace",
    index=False
)

print("Silver records:", len(silver_orders))

display(silver_orders)


Silver records: 3


,order_id,customer_id,product_id,quantity,price,rejection_reason,total_amount
0,O101,C001,P001,2,15000.0,,30000.0
1,O102,C002,P002,1,45000.0,,45000.0
6,O106,C004,P006,3,800.0,,2400.0


In [61]:

gold_summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_orders,
    SUM(total_amount) AS total_revenue,
    ROUND(AVG(total_amount), 2) AS average_order_value
FROM silver_orders;
""", connection)

gold_summary.to_sql(
    "gold_sales_summary",
    connection,
    if_exists="replace",
    index=False
)

print("Gold summary created successfully!")

display(gold_summary)


Gold summary created successfully!


,total_orders,total_revenue,average_order_value
0,3,77400.0,25800.0


In [62]:

# Create a separate table for our UPSERT demonstration

connection.execute("""
CREATE TABLE IF NOT EXISTS upsert_orders (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    product_id TEXT,
    quantity INTEGER,
    price REAL,
    total_amount REAL
)
""")

# Reset only our demonstration table
connection.execute("DELETE FROM upsert_orders")

# Insert the original order
connection.execute("""
INSERT INTO upsert_orders
VALUES ('O107', 'C002', 'P006', 2, 800, 1600)
""")

connection.commit()

print("Initial order created successfully!")

display(pd.read_sql_query(
    "SELECT * FROM upsert_orders",
    connection
))


Initial order created successfully!


,order_id,customer_id,product_id,quantity,price,total_amount
0,O107,C002,P006,2,800.0,1600.0


In [63]:

connection.execute("""
INSERT INTO upsert_orders
    (order_id, customer_id, product_id, quantity, price, total_amount)
VALUES
    ('O107', 'C002', 'P006', 5, 800, 4000)

ON CONFLICT(order_id)
DO UPDATE SET
    customer_id = excluded.customer_id,
    product_id = excluded.product_id,
    quantity = excluded.quantity,
    price = excluded.price,
    total_amount = excluded.total_amount
""")

connection.commit()

print("Order updated successfully!")

display(pd.read_sql_query(
    "SELECT * FROM upsert_orders",
    connection
))


Order updated successfully!


,order_id,customer_id,product_id,quantity,price,total_amount
0,O107,C002,P006,5,800.0,4000.0


In [64]:

connection.execute("""
INSERT INTO upsert_orders
    (order_id, customer_id, product_id, quantity, price, total_amount)
VALUES
    ('O108', 'C003', 'P005', 3, 1500, 4500)

ON CONFLICT(order_id)
DO UPDATE SET
    quantity = excluded.quantity,
    price = excluded.price,
    total_amount = excluded.total_amount
""")

connection.commit()

display(pd.read_sql_query(
    "SELECT * FROM upsert_orders ORDER BY order_id",
    connection
))


,order_id,customer_id,product_id,quantity,price,total_amount
0,O107,C002,P006,5,800.0,4000.0
1,O108,C003,P005,3,1500.0,4500.0


In [65]:

# Create a separate watermark demonstration

connection.execute("DROP TABLE IF EXISTS watermark_source")
connection.execute("DROP TABLE IF EXISTS watermark_target")
connection.execute("DROP TABLE IF EXISTS watermark_state")

# Incoming source data
connection.execute("""
CREATE TABLE watermark_source (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    total_amount REAL,
    created_at TEXT
)
""")

# Destination table
connection.execute("""
CREATE TABLE watermark_target (
    order_id TEXT PRIMARY KEY,
    customer_id TEXT,
    total_amount REAL,
    created_at TEXT
)
""")

# Store the last successfully processed timestamp
connection.execute("""
CREATE TABLE watermark_state (
    pipeline_name TEXT PRIMARY KEY,
    last_watermark TEXT
)
""")

# Insert first three incoming orders
connection.executemany("""
INSERT INTO watermark_source
VALUES (?, ?, ?, ?)
""", [
    ("O201", "C001", 3000, "2026-11-19 09:00:00"),
    ("O202", "C002", 4500, "2026-11-19 10:00:00"),
    ("O203", "C003", 2500, "2026-11-19 11:00:00")
])

# Start watermark before the first order
connection.execute("""
INSERT INTO watermark_state
VALUES ('retail_pipeline', '2026-11-19 08:00:00')
""")

connection.commit()

print("Watermark demonstration initialized!")


Watermark demonstration initialized!


In [66]:

def watermark_incremental_load(connection):

    logger.info("Watermark pipeline started")

    # 1. Read the previous watermark
    watermark = pd.read_sql_query("""
        SELECT last_watermark
        FROM watermark_state
        WHERE pipeline_name = 'retail_pipeline'
    """, connection)["last_watermark"].iloc[0]

    logger.info("Previous watermark: %s", watermark)

    # 2. Extract only records newer than the watermark
    new_data = pd.read_sql_query("""
        SELECT *
        FROM watermark_source
        WHERE created_at > ?
        ORDER BY created_at
    """, connection, params=(watermark,))

    if new_data.empty:
        logger.info("No new records found")
        return new_data

    # 3. Load the new records
    new_data.to_sql(
        "watermark_target",
        connection,
        if_exists="append",
        index=False
    )

    # 4. Update watermark after successful loading
    new_watermark = new_data["created_at"].max()

    connection.execute("""
        UPDATE watermark_state
        SET last_watermark = ?
        WHERE pipeline_name = 'retail_pipeline'
    """, (new_watermark,))

    connection.commit()

    logger.info(
        "Successfully loaded %s records",
        len(new_data)
    )

    logger.info("Updated watermark: %s", new_watermark)

    return new_data

In [68]:

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    force=True
)

logger = logging.getLogger("RetailPipeline")

print("Logger initialized successfully!")


Logger initialized successfully!


In [69]:

first_batch = watermark_incremental_load(connection)

print("First batch records:", len(first_batch))

display(pd.read_sql_query(
    "SELECT * FROM watermark_target ORDER BY created_at",
    connection
))


2026-11-19 09:09:30,311 - INFO - Watermark pipeline started
2026-11-19 09:09:30,315 - INFO - Previous watermark: 2026-11-19 08:00:00
2026-11-19 09:09:30,343 - INFO - Successfully loaded 3 records
2026-11-19 09:09:30,344 - INFO - Updated watermark: 2026-11-19 11:00:00


First batch records: 3


,order_id,customer_id,total_amount,created_at
0,O201,C001,3000.0,2026-11-19 09:00:00
1,O202,C002,4500.0,2026-11-19 10:00:00
2,O203,C003,2500.0,2026-11-19 11:00:00


In [70]:

# Check total records loaded
target_data = pd.read_sql_query(
    "SELECT * FROM watermark_target ORDER BY created_at",
    connection
)

# Check current watermark
watermark_data = pd.read_sql_query(
    "SELECT * FROM watermark_state",
    connection
)

print("Total records in target:", len(target_data))
print("\nLoaded orders:")
display(target_data)

print("\nCurrent watermark:")
display(watermark_data)


Total records in target: 3

Loaded orders:


,order_id,customer_id,total_amount,created_at
0,O201,C001,3000.0,2026-11-19 09:00:00
1,O202,C002,4500.0,2026-11-19 10:00:00
2,O203,C003,2500.0,2026-11-19 11:00:00



Current watermark:


,pipeline_name,last_watermark
0,retail_pipeline,2026-11-19 11:00:00


In [71]:

def check_data_quality(data):

    results = {}

    # 1. Completeness
    results["missing_values"] = (
        data.isnull().sum().sum()
    )

    # 2. Uniqueness
    results["duplicate_order_ids"] = (
        data["order_id"].duplicated().sum()
    )

    # 3. Validity
    results["invalid_quantities"] = (
        (data["quantity"].notna()) &
        (data["quantity"] <= 0)
    ).sum()

    results["negative_prices"] = (
        (data["price"].notna()) &
        (data["price"] < 0)
    ).sum()

    return results


quality_report = check_data_quality(raw_orders)

print("DATA QUALITY REPORT")

for check, result in quality_report.items():
    print(check, ":", result)


DATA QUALITY REPORT
missing_values : 2
duplicate_order_ids : 1
invalid_quantities : 1
negative_prices : 0


In [72]:

def generate_quality_report(data):

    report = check_data_quality(data)

    total_records = len(data)

    # Count records with at least one issue
    invalid_records = (
        data.isnull().any(axis=1)
        | data.duplicated(subset=["order_id"])
        | (data["quantity"].notna() & (data["quantity"] <= 0))
        | (data["price"].notna() & (data["price"] < 0))
    ).sum()

    valid_records = total_records - invalid_records

    quality_percentage = (
        (valid_records / total_records) * 100
        if total_records > 0 else 0
    )

    report["total_records"] = total_records
    report["valid_records"] = valid_records
    report["invalid_records"] = invalid_records
    report["quality_percentage"] = round(
        quality_percentage, 2
    )

    return report


final_report = generate_quality_report(raw_orders)

print("FINAL DATA QUALITY REPORT")

for check, result in final_report.items():
    print(check, ":", result)


FINAL DATA QUALITY REPORT
missing_values : 2
duplicate_order_ids : 1
invalid_quantities : 1
negative_prices : 0
total_records : 7
valid_records : 3
invalid_records : 4
quality_percentage : 42.86
